# BorsaBot — Model Eğitim Pipeline (Google Colab)
**Veri:** Yahoo Finance (yfinance) — Türkiye dahil her yerden çalışır
**Modeller:** HMM Rejim + XGBoost Primary + LightGBM Meta
**Backtest:** CPCV (C(6,2)=15 path)

In [ ]:
# ──────────────────────────────────────────────────────────────────
!pip install -q yfinance xgboost lightgbm hmmlearn scikit-learn imbalanced-learn pandas numpy pyarrow tqdm scipy matplotlib

import os, time, math, itertools, pickle, warnings, shutil
from pathlib import Path

import yfinance as yf
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy.stats import norm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from hmmlearn.hmm import GaussianHMM

warnings.filterwarnings("ignore")
print("✓ Tüm paketler yüklendi")

In [ ]:
# ──────────────────────────────────────────────────────────────────
# Yahoo Finance sembolleri: BTC-USD, ETH-USD, BNB-USD, SOL-USD
# Interval seçenekleri:
#   "1d"  → Günlük  (tüm tarih aralığı desteklenir)
#   "1h"  → Saatlik (sadece son 730 gün — 2021 öncesi veri yok)
#   "1wk" → Haftalık

CFG = dict(
    # Yahoo Finance sembolleri (BTCUSDT → BTC-USD)
    symbols    = ["BTC-USD", "ETH-USD"],
    interval   = "1d",          # Günlük → 10 yıla kadar geçmiş
    start_date = "2018-01-01",  # 1d için herhangi bir tarih
    end_date   = "2024-12-31",

    # Triple Barrier parametreleri
    vertical_bars = 20,     # 20 gün maks. tutma (1d için)
    pt_sl         = [1.5, 1.0],
    min_ret       = 0.005,  # min %0.5 getiri (günlük için daha yüksek)

    # CPCV
    n_groups   = 6,
    n_test     = 2,
    embargo_pct= 0.01,
    fee_bps    = 10.0,   # günlük bar → daha az işlem → daha az spread gerekir

    # Saklama
    raw_dir    = "/content/borsabot_data/raw",
    out_dir    = "/content/borsabot_data/processed",
    model_dir  = "/content/borsabot_data/models",
)

LABEL_MAP  = {-1: 0, 0: 1, 1: 2}
LABEL_RMAP = {0: -1, 1: 0, 2: 1}

for d in [CFG["raw_dir"], CFG["out_dir"], CFG["model_dir"]]:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f"✓ Konfigürasyon hazır")
print(f"  Semboller : {CFG['symbols']}")
print(f"  Interval  : {CFG['interval']}")
print(f"  Tarih     : {CFG['start_date']} → {CFG['end_date']}")

In [ ]:
# ──────────────────────────────────────────────────────────────────
def download_ohlcv(sym: str, interval: str,
                   start_date: str, end_date: str,
                   raw_dir: str) -> pd.DataFrame:
    """
    yfinance ile OHLCV verisi indir.
    Hourly (1h) için yfinance son 730 günle sınırlıdır.
    Daily (1d) için tüm tarih aralığını destekler.
    """
    path = Path(raw_dir) / f"{sym.replace('-','_')}_{interval}.parquet"
    if path.exists():
        print(f"  [{sym}] Cache'den yükleniyor ...")
        return pd.read_parquet(path)

    print(f"  [{sym}] yfinance'den indiriliyor ...")

    if interval == "1h":
        # Saatlik: 730 günlük pencerelerle çek
        frames = []
        end_dt   = pd.Timestamp(end_date)
        start_dt = pd.Timestamp(start_date)
        cursor   = max(start_dt, end_dt - pd.Timedelta(days=729))

        while cursor <= end_dt:
            chunk_end = min(cursor + pd.Timedelta(days=728), end_dt)
            tk = yf.Ticker(sym)
            df_chunk = tk.history(
                start=cursor.strftime("%Y-%m-%d"),
                end=chunk_end.strftime("%Y-%m-%d"),
                interval="1h",
                auto_adjust=True,
            )
            if not df_chunk.empty:
                frames.append(df_chunk[["Open","High","Low","Close","Volume"]])
            cursor = chunk_end + pd.Timedelta(hours=1)
            time.sleep(0.5)

        if not frames:
            raise RuntimeError(f"{sym} için saatlik veri bulunamadı. "
                               f"interval='1d' deneyin.")
        df = pd.concat(frames).sort_index()
    else:
        # Günlük / Haftalık / Aylık — tüm tarih aralığı
        tk = yf.Ticker(sym)
        df = tk.history(
            start=start_date,
            end=end_date,
            interval=interval,
            auto_adjust=True,
        )[["Open","High","Low","Close","Volume"]]

    # Kolon isimlerini küçük harfe çevir
    df.columns = ["open","high","low","close","volume"]

    # Timezone'u kaldır (consistent index için)
    if df.index.tz is not None:
        df.index = df.index.tz_localize(None)

    df = df[~df.index.duplicated(keep="first")].sort_index()
    df = df.dropna(subset=["close"])
    df.to_parquet(path)
    print(f"  [{sym}] {len(df):,} bar kaydedildi → {path.name}")
    return df

In [ ]:
# ──────────────────────────────────────────────────────────────────
ohlcv: dict[str, pd.DataFrame] = {}

for sym in CFG["symbols"]:
    ohlcv[sym] = download_ohlcv(
        sym, CFG["interval"],
        CFG["start_date"], CFG["end_date"],
        CFG["raw_dir"],
    )

for sym, df in ohlcv.items():
    print(f"\n{sym}: {len(df):,} bar  "
          f"| {df.index[0].date()} → {df.index[-1].date()}")
    print(f"  Close: ${df['close'].iloc[-1]:,.2f}  "
          f"Vol: {df['volume'].iloc[-1]:,.0f}")

In [ ]:
# ──────────────────────────────────────────────────────────────────
def ffd_weights(d: float, thres: float = 1e-5) -> np.ndarray:
    w = [1.0]; k = 1
    while abs(w[-1]) >= thres:
        w.append(-w[-1] * (d - k + 1) / k); k += 1
    return np.array(w[::-1])

def frac_diff(series: pd.Series, d: float = 0.4) -> pd.Series:
    """Sabit-pencere Fractional Differentiation."""
    w = ffd_weights(d); width = len(w); out = {}
    for i in range(width - 1, len(series)):
        out[series.index[i]] = float(np.dot(w, series.iloc[i-width+1:i+1].values))
    return pd.Series(out, name="fd_close")

def compute_features(df: pd.DataFrame, d: float = 0.4,
                     interval: str = "1d") -> pd.DataFrame:
    """14 özellik üretir — interval'e göre pencere boyutları ayarlanır."""
    # Günlük için pencereler: 14, 24, 72 bar
    # Saatlik için aynı sayılar zaten saate karşılık gelir
    w_short = 14   # RSI, ATR penceresi
    w_mid   = 24   # Vol, OBV, BB, VWAP
    w_long  = 72   # Uzun vol, OBV

    f = pd.DataFrame(index=df.index)

    # Frac-diff log kapanış
    f["fd_close"] = frac_diff(np.log(df["close"]), d)

    # Gecikmeli getiriler
    ret = df["close"].pct_change()
    for lag in [1, 2, 3, 5, 10, 20]:
        f[f"ret_{lag}"] = ret.shift(lag)

    # Volatilite
    f[f"vol_{w_mid}"] = ret.rolling(w_mid).std()
    f[f"vol_{w_long}"] = ret.rolling(w_long).std()

    # RSI
    delta = df["close"].diff()
    gain  = delta.clip(lower=0).ewm(com=w_short-1, min_periods=w_short).mean()
    loss  = (-delta.clip(upper=0)).ewm(com=w_short-1, min_periods=w_short).mean()
    f["rsi_14"] = 100 - 100 / (1 + gain / (loss + 1e-9))

    # ATR (normalize)
    hl  = df["high"] - df["low"]
    hpc = (df["high"] - df["close"].shift(1)).abs()
    lpc = (df["low"]  - df["close"].shift(1)).abs()
    tr  = pd.concat([hl, hpc, lpc], axis=1).max(axis=1)
    f["atr_14"] = tr.ewm(com=w_short-1, min_periods=w_short).mean() / df["close"]

    # OBV normalize
    obv = (df["volume"] * np.sign(df["close"].diff().fillna(0))).cumsum()
    f["obv_norm"] = (obv - obv.rolling(w_long).mean()) / (obv.rolling(w_long).std() + 1e-9)

    # Bollinger Band genişliği
    ma   = df["close"].rolling(20).mean()
    std  = df["close"].rolling(20).std()
    f["bb_width"] = 2 * std / (ma + 1e-9)

    # VWAP sapması
    tp   = (df["high"] + df["low"] + df["close"]) / 3
    vwap = (tp * df["volume"]).rolling(w_mid).sum() / (df["volume"].rolling(w_mid).sum() + 1e-9)
    f["close_vwap"] = (df["close"] - vwap) / (df["close"].rolling(w_mid).std() + 1e-9)

    # Log hacim
    f["log_vol"] = np.log(df["volume"] / (df["volume"].rolling(w_mid).mean() + 1e-9) + 1e-9)

    return f.dropna()

In [ ]:
# ──────────────────────────────────────────────────────────────────
features: dict[str, pd.DataFrame] = {}
for sym, df in ohlcv.items():
    feat = compute_features(df, interval=CFG["interval"])
    feat.to_parquet(f"{CFG['out_dir']}/{sym.replace('-','_')}_features.parquet")
    features[sym] = feat
    print(f"[{sym}] {feat.shape[0]:,} bar × {feat.shape[1]} özellik")

FEAT_COLS = list(features[CFG["symbols"][0]].columns)
print(f"\n✓ Özellikler: {FEAT_COLS}")

In [ ]:
# ──────────────────────────────────────────────────────────────────
def ewm_vol(close: pd.Series, span: int = 100) -> pd.Series:
    return close.pct_change().dropna().ewm(span=span, min_periods=span).std()

def triple_barrier(close: pd.Series, trgt: pd.Series,
                   pt_sl=(1.5, 1.0), vertical_bars=20,
                   min_ret=0.005) -> pd.DataFrame:
    trgt = trgt.dropna()
    if min_ret > 0:
        trgt = trgt[trgt > min_ret]

    records = []
    idx = close.index
    for t0 in tqdm(trgt.index, desc="  Triple Barrier", leave=False):
        if t0 not in idx:
            continue
        loc    = idx.get_loc(t0)
        t1_loc = min(loc + vertical_bars, len(idx) - 1)
        t1     = idx[t1_loc]
        path   = close.loc[t0:t1]
        ret    = (path / close.loc[t0]) - 1.0
        tgt_v  = trgt.loc[t0]
        pt     =  tgt_v * pt_sl[0]
        sl     = -tgt_v * pt_sl[1]

        up_hits   = ret[ret >= pt]
        down_hits = ret[ret <= sl]
        up_t      = up_hits.index[0]   if len(up_hits)   > 0 else pd.NaT
        down_t    = down_hits.index[0] if len(down_hits) > 0 else pd.NaT

        if pd.isna(up_t) and pd.isna(down_t):
            label = 0
        elif pd.isna(down_t) or (not pd.isna(up_t) and up_t <= down_t):
            label = 1
        else:
            label = -1

        records.append({"t0": t0, "t1": t1, "trgt": tgt_v, "label": label})

    return pd.DataFrame(records).set_index("t0")

In [ ]:
# ──────────────────────────────────────────────────────────────────
labeled: dict[str, pd.DataFrame] = {}

for sym in CFG["symbols"]:
    close = ohlcv[sym]["close"]
    vol   = ewm_vol(close, span=100).reindex(features[sym].index).dropna()

    df_ev = triple_barrier(
        close,
        vol,
        pt_sl         = CFG["pt_sl"],
        vertical_bars = CFG["vertical_bars"],
        min_ret       = CFG["min_ret"],
    )

    df_lbl = features[sym].loc[df_ev.index].copy()
    df_lbl["label"] = df_ev["label"]
    df_lbl["t1"]    = df_ev["t1"]
    df_lbl["trgt"]  = df_ev["trgt"]
    df_lbl = df_lbl.dropna(subset=["label"])

    labeled[sym] = df_lbl
    df_lbl.to_parquet(f"{CFG['out_dir']}/{sym.replace('-','_')}_labeled.parquet")

    d = df_lbl["label"].value_counts().sort_index()
    n = len(df_lbl)
    print(f"\n[{sym}] {n:,} etiket")
    for lbl, name in [(-1,"SELL"), (0,"FLAT"), (1,"BUY")]:
        cnt = d.get(lbl, 0)
        print(f"  {name:4s}: {cnt:5,}  ({cnt/n*100:.1f}%)")

print("\n✓ Etiketleme tamamlandı")

In [ ]:
# ──────────────────────────────────────────────────────────────────
all_close = pd.concat([ohlcv[s]["close"] for s in CFG["symbols"]]).sort_index()
ret_all   = all_close.pct_change().dropna().values
obs_all   = np.column_stack([ret_all, np.abs(ret_all), np.sign(ret_all)])

hmm = GaussianHMM(n_components=3, covariance_type="full",
                  n_iter=200, random_state=42)
hmm.fit(obs_all)
states   = hmm.predict(obs_all)
mean_vol = {s: float(np.mean(np.abs(ret_all[states == s]))) for s in range(3)}
sorted_s = sorted(mean_vol, key=mean_vol.get)
regime_mapping = {sorted_s[0]: "low_vol",
                  sorted_s[1]: "trending",
                  sorted_s[2]: "high_vol"}

with open(f"{CFG['model_dir']}/regime.pkl", "wb") as f:
    pickle.dump({"model": hmm, "mapping": regime_mapping}, f)

print("✓ Rejim Dedektörü (HMM) eğitildi")
print(f"  Durum eşlemesi: {regime_mapping}")

In [ ]:
# ──────────────────────────────────────────────────────────────────
primary_models: dict[str, XGBClassifier] = {}

for sym in CFG["symbols"]:
    df = labeled[sym]
    X  = df[FEAT_COLS].values
    y  = df["label"].map(LABEL_MAP).values

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.15, shuffle=False)

    try:
        sm = SMOTE(random_state=42, k_neighbors=min(3, min(np.bincount(y_tr))-1))
        X_tr, y_tr = sm.fit_resample(X_tr, y_tr)
        print(f"[{sym}] SMOTE sonrası: {len(X_tr):,} sample")
    except Exception as e:
        print(f"[{sym}] SMOTE atlandı: {e}")

    clf = XGBClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
        reg_alpha=0.1, reg_lambda=1.0,
        num_class=3, objective="multi:softprob",
        eval_metric="mlogloss", random_state=42,
        n_jobs=-1, verbosity=0,
    )
    clf.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)
    primary_models[sym] = clf

    print(f"\n[{sym}] Primary Model (XGBoost):")
    print(classification_report(y_te, clf.predict(X_te),
        target_names=["SELL(-1)", "FLAT(0)", "BUY(+1)"], zero_division=0))

    safe_sym = sym.replace("-", "_")
    with open(f"{CFG['model_dir']}/{safe_sym}_primary.pkl", "wb") as f:
        pickle.dump({"model": clf, "feat_cols": FEAT_COLS,
                     "label_map": LABEL_MAP, "label_rmap": LABEL_RMAP,
                     "symbol": sym, "interval": CFG["interval"]}, f)
    print(f"  → {safe_sym}_primary.pkl kaydedildi")

In [ ]:
# ──────────────────────────────────────────────────────────────────
meta_models: dict[str, LGBMClassifier] = {}

for sym in CFG["symbols"]:
    df    = labeled[sym]
    X     = df[FEAT_COLS].values
    y     = df["label"].map(LABEL_MAP).values
    pred  = primary_models[sym].predict(X)
    proba = primary_models[sym].predict_proba(X)
    conf  = proba.max(axis=1)
    side  = np.vectorize(LABEL_RMAP.get)(pred).astype(float)

    # FLAT olmayan sample'lar
    non_flat = pred != 1
    X_meta   = np.column_stack([X[non_flat], side[non_flat], conf[non_flat]])
    y_meta   = (pred[non_flat] == y[non_flat]).astype(int)

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_meta, y_meta, test_size=0.15, shuffle=False)

    pos_w = max(1.0, (y_tr == 0).sum() / ((y_tr == 1).sum() + 1e-9))

    meta_clf = LGBMClassifier(
        n_estimators=400, num_leaves=31, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
        reg_alpha=0.1, reg_lambda=1.0, scale_pos_weight=pos_w,
        random_state=42, n_jobs=-1, verbose=-1,
    )
    meta_clf.fit(X_tr, y_tr, eval_set=[(X_te, y_te)])
    meta_models[sym] = meta_clf

    print(f"\n[{sym}] Meta Model (LightGBM):")
    print(classification_report(y_te, meta_clf.predict(X_te),
        target_names=["WRONG", "CORRECT"], zero_division=0))

    meta_feat_cols = FEAT_COLS + ["primary_pred", "primary_conf"]
    safe_sym = sym.replace("-", "_")
    with open(f"{CFG['model_dir']}/{safe_sym}_meta.pkl", "wb") as f:
        pickle.dump({"model": meta_clf, "feat_cols": meta_feat_cols,
                     "symbol": sym}, f)
    print(f"  → {safe_sym}_meta.pkl kaydedildi")

print("\n✓ Tüm modeller eğitildi!")

In [ ]:
# ──────────────────────────────────────────────────────────────────
def cpcv_splits(n, n_groups, n_test, embargo_pct):
    borders = [i * (n // n_groups) for i in range(n_groups)] + [n]
    groups  = [list(range(borders[i], borders[i+1])) for i in range(n_groups)]
    emb_n   = max(1, int(n * embargo_pct))
    for test_ids in itertools.combinations(range(n_groups), n_test):
        test_s  = set(j for g in test_ids for j in groups[g])
        train_s = set(j for g in range(n_groups) if g not in test_ids
                      for j in groups[g])
        emb = {p - o for p in test_s for o in range(1, emb_n + 1)
               if (p - o) >= 0 and (p - o) not in test_s}
        train_s -= emb
        if train_s and test_s:
            yield np.array(sorted(train_s)), np.array(sorted(test_s))

def simulate(signals, prices, fee_bps):
    cap = 100_000.0; pos = 0; equit = [cap]; rets = []
    cost = fee_bps / 10_000
    for i in range(1, len(signals)):
        pp, pn = prices[i-1], prices[i]
        sig = signals[i]
        if pos != 0 and pp > 0:
            cap *= (1 + (pn - pp) / pp * pos)
        if sig != pos:
            cap *= (1 - abs(sig - pos) * cost); pos = int(sig)
        equit.append(cap)
        rets.append(cap / equit[-2] - 1 if equit[-2] > 0 else 0.0)
    eq = np.array(equit); r = np.array(rets)
    periods = 365 if CFG["interval"] == "1d" else 365 * 24
    sr = r.mean() / (r.std() + 1e-9) * np.sqrt(periods)
    rm = np.maximum.accumulate(eq)
    return {"sharpe": float(sr),
            "mdd":    float(((eq - rm) / (rm + 1e-9)).min()),
            "equity": eq}

In [ ]:
# ──────────────────────────────────────────────────────────────────
META_THRESHOLD = 0.55

for sym in CFG["symbols"]:
    df      = labeled[sym]
    X_all   = df[FEAT_COLS].values
    y_all   = df["label"].map(LABEL_MAP).values
    prices  = ohlcv[sym]["close"].reindex(df.index).ffill().values
    prim    = primary_models[sym]
    meta    = meta_models[sym]

    sharpes, mdds, equities = [], [], []

    splits = list(cpcv_splits(len(df), CFG["n_groups"],
                               CFG["n_test"], CFG["embargo_pct"]))

    for tr_idx, te_idx in tqdm(splits, desc=f"[{sym}] CPCV"):
        pf = XGBClassifier(**{**prim.get_params(), "verbosity": 0})
        try:
            pf.fit(X_all[tr_idx], y_all[tr_idx])
        except Exception:
            continue

        X_te  = X_all[te_idx]
        pred  = pf.predict(X_te)
        proba = pf.predict_proba(X_te)
        conf  = proba.max(axis=1)
        side  = np.vectorize(LABEL_RMAP.get)(pred).astype(float)

        X_meta = np.column_stack([X_te, side, conf])
        m_prob = meta.predict_proba(X_meta)[:, 1]
        sigs   = np.where(m_prob >= META_THRESHOLD, side, 0.0)

        res = simulate(sigs, prices[te_idx], CFG["fee_bps"])
        sharpes.append(res["sharpe"]); mdds.append(res["mdd"])
        equities.append(res["equity"])

    sarr = np.array(sharpes)
    psr  = float(norm.cdf(sarr.mean() / (sarr.std() + 1e-9)))

    print(f"\n{'═'*55}")
    print(f"  {sym} — CPCV ({len(sarr)} path)")
    print(f"{'═'*55}")
    print(f"  Ortalama Sharpe : {sarr.mean():+.3f}  ± {sarr.std():.3f}")
    print(f"  En İyi / Kötü   : {sarr.max():+.3f}  / {sarr.min():+.3f}")
    print(f"  Ortalama MDD    : {np.mean(mdds)*100:.1f}%")
    print(f"  PSR             : {psr:.3f}  {'✓ iyi' if psr > 0.55 else '✗ düşük'}")

    # Görselleştirme
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.hist(sarr, bins=min(15, len(sarr)), color="#4C72B0", edgecolor="white")
    ax1.axvline(0, color="red", ls="--", label="SR=0")
    ax1.axvline(sarr.mean(), color="lime", ls="-", label=f"Ort={sarr.mean():.2f}")
    ax1.set_title(f"{sym} — Sharpe Dağılımı  (PSR={psr:.2f})")
    ax1.set_xlabel("Sharpe (yıllık)"); ax1.legend()
    for eq in equities:
        ax2.plot(eq / eq[0], alpha=0.35, lw=0.8)
    ax2.axhline(1.0, color="gray", ls="--")
    ax2.set_title(f"{sym} — CPCV Equity Paths")
    ax2.set_xlabel("Bar"); ax2.set_ylabel("Büyüme")
    plt.tight_layout()
    safe_sym = sym.replace("-", "_")
    plt.savefig(f"{CFG['out_dir']}/{safe_sym}_cpcv.png", dpi=90)
    plt.show()
    print(f"  Grafik: {CFG['out_dir']}/{safe_sym}_cpcv.png")

In [ ]:
# ──────────────────────────────────────────────────────────────────
shutil.make_archive("/content/borsabot_models", "zip", CFG["model_dir"])

print("\n" + "═"*50)
print("✓ Eğitim tamamlandı!")
print("─"*50)
print("ZIP: /content/borsabot_models.zip")
print("→ Sol panelde Files sekmesi → dosyaya sağ tık → Download")
print("─"*50)
print("Yerel kullanım:")
print("  1. ZIP'i aç → c:\\borsaBot\\models\\ klasörüne koy")
print("  2. python scripts/main.py --broker mock --paper --model-dir models/")
print("═"*50)

# Google Drive'a kaydet (opsiyonel)
# from google.colab import drive
# drive.mount("/content/drive")
# shutil.copytree(CFG["model_dir"],
#                 "/content/drive/MyDrive/borsabot_models",
#                 dirs_exist_ok=True)